In [22]:
!pip install sentence-transformers
!pip install scikit-learn
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sentence_transformers import SentenceTransformer


In [32]:
df = pd.read_csv("symptoms.csv")
df.head()


,text,outcome,stage
0,"thấy hay buồn ngủ, đặc biệt là buồn ngủ sau kh...",1,1
1,"cảm thấy mệt mỏi không có sức, chắc do làm việ...",0,0
2,vết thương ở chân của tôi mấy tuần rồi mà khôn...,1,3
3,"Tôi hay đói, ăn nhiều nhưng vẫn tụt kg",1,2
4,"tôi bị đau đầu và sổ mũi, có vẻ như bị cảm",0,0


In [24]:
model = SentenceTransformer('VoVanPhuc/sup-SimCSE-VietNamese-phobert-base')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/17.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [25]:
X_text = df["text"].tolist()
X_embeddings = model.encode(X_text)

y_outcome = df["outcome"]
y_stage = df["stage"]


In [26]:
outcome_clf = LogisticRegression(max_iter=2000)
outcome_clf.fit(X_embeddings, y_outcome)

print("Đánh giá outcome:")
print(classification_report(y_outcome, outcome_clf.predict(X_embeddings)))


Đánh giá outcome:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       220
           1       1.00      1.00      1.00       280

    accuracy                           1.00       500
   macro avg       1.00      1.00      1.00       500
weighted avg       1.00      1.00      1.00       500



In [27]:
df_diab = df[df["outcome"] == 1]

X_stage_embed = model.encode(df_diab["text"].tolist())
stage_clf = LogisticRegression(max_iter=2000)
stage_clf.fit(X_stage_embed, df_diab["stage"])

print("Đánh giá stage:")
print(classification_report(df_diab["stage"], stage_clf.predict(X_stage_embed)))


Đánh giá stage:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        97
           2       0.99      1.00      0.99        90
           3       1.00      0.99      0.99        93

    accuracy                           1.00       280
   macro avg       1.00      1.00      1.00       280
weighted avg       1.00      1.00      1.00       280



In [28]:
def phobert_predict(text):
    emb = model.encode([text])

    # 1. Outcome
    outcome = outcome_clf.predict(emb)[0]

    if outcome == 0:
        return {
            "outcome": 0,
            "stage": 0,
            "answer": "Dựa trên mô tả, bạn không có dấu hiệu rõ ràng của bệnh tiểu đường."
        }

    # 2. Stage
    stage = stage_clf.predict(emb)[0]

    stage_text = {
        1: "Bạn có dấu hiệu nhẹ, nghi ngờ tiểu đường giai đoạn đầu.",
        2: "Triệu chứng rõ rệt hơn, có thể thuộc giai đoạn 2.",
        3: "Triệu chứng của bạn thuộc nhóm nặng hơn, nên kiểm tra sớm."
    }

    return {
        "outcome": 1,
        "stage": int(stage),
        "answer": stage_text[stage]
    }


In [37]:
user_input = input("Nhập mô tả của bạn: ")
result = phobert_predict(user_input)
result


Nhập mô tả của bạn: tôi muốn đi ăn


{'outcome': 0,
 'stage': 0,
 'answer': 'Dựa trên mô tả, bạn không có dấu hiệu rõ ràng của bệnh tiểu đường.'}